# Apple Detection — YOLO Training

### Before you start
1. Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU → Save**
2. Run each cell top to bottom (or **Runtime → Run all**)
3. When Cell 3 runs, a file picker will appear — upload your `dataset.zip` from your PC
4. After training, Cell 7 will automatically download `best.pt` to your PC

In [ ]:
# Cell 1 — Check GPU
import torch
if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 2 — Install dependencies
!pip install -q ultralytics
print("Done.")

In [ ]:
# Cell 3 — Upload dataset.zip
# A file picker will appear below. Select dataset.zip from your PC.
from google.colab import files
import os

print("Select dataset.zip when the picker appears...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"Uploaded: {zip_name} ({os.path.getsize(zip_name) / 1e6:.1f} MB)")

In [ ]:
# Cell 4 — Unzip and verify dataset
import zipfile, pathlib

DATASET_DIR = pathlib.Path("/content/dataset")
DATASET_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)

# Find data.yaml (handles nested zip structures)
yaml_files = list(DATASET_DIR.rglob("data.yaml"))
if not yaml_files:
    raise FileNotFoundError("data.yaml not found in zip. Make sure you uploaded dataset.zip from the Pi.")

DATA_YAML = yaml_files[0]
DATA_ROOT = DATA_YAML.parent
print(f"data.yaml found at: {DATA_YAML}")

# Count images
for split in ["train", "valid", "test"]:
    imgs = list((DATA_ROOT / split / "images").glob("*.jpg"))
    print(f"  {split}: {len(imgs)} images")

In [ ]:
# Cell 5 — Fix data.yaml paths to absolute (required for Colab)
import yaml

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

data_cfg["train"] = str(DATA_ROOT / "train" / "images")
data_cfg["val"]   = str(DATA_ROOT / "valid" / "images")
data_cfg["test"]  = str(DATA_ROOT / "test"  / "images")

with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f)

print("data.yaml updated:")
print(f"  train : {data_cfg['train']}")
print(f"  val   : {data_cfg['val']}")
print(f"  test  : {data_cfg['test']}")
print(f"  classes ({data_cfg['nc']}): {data_cfg['names']}")

In [ ]:
# Cell 6 — Train
# Adjust EPOCHS and MODEL if you like:
#   Models (fastest to most accurate): yolov8n  yolov8s  yolov8m
EPOCHS = 50
MODEL  = "yolov8n"   # nano — best for Raspberry Pi inference speed
IMGSZ  = 640
BATCH  = 16

from ultralytics import YOLO

model = YOLO(f"{MODEL}.pt")
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project="/content/runs",
    name="apple",
    exist_ok=True,
    verbose=True,
)

BEST_WEIGHTS = pathlib.Path(results.save_dir) / "weights" / "best.pt"
print(f"\nTraining complete. Best weights: {BEST_WEIGHTS}")

In [ ]:
# Cell 7 — Validate (quick accuracy check)
val_model = YOLO(str(BEST_WEIGHTS))
metrics = val_model.val(data=str(DATA_YAML), imgsz=IMGSZ, verbose=False)

print("\n=== Validation Results ===")
print(f"  mAP50     : {metrics.box.map50:.4f}")
print(f"  mAP50-95  : {metrics.box.map:.4f}")
print(f"  Precision : {metrics.box.mp:.4f}")
print(f"  Recall    : {metrics.box.mr:.4f}")

In [ ]:
# Cell 8 — Download best.pt to your PC
# The file will appear in your browser's Downloads folder.
from google.colab import files
files.download(str(BEST_WEIGHTS))
print("best.pt downloaded!")
print()
print("Next steps on the Pi:")
print("  1. Copy best.pt to ~/yolo-project/weights/best.pt")
print("  2. Edit config/model.yaml → set  weights: weights/best.pt")
print("  3. Run: python inference.py --device 0 --headless")